# Model training
This use-case is model training.  
By going through this you will know how to use Cascade for metadata tracking, hyperparameter tuning and model selection.  
  
Previous part is the pipeline building and is taken without comments.  
For more detailed description of it see Pipeline building example.

In [32]:
#!pip3 install torchvision

In [33]:
import cascade.data as cdd
from cascade.utils.torch import TorchModel
from cascade.utils.sklearn import SkMetric

from tqdm import tqdm
import torch
import torchvision
from torchvision.transforms import functional as F
from torch import nn

In [34]:
import cascade
cascade.__version__

'0.18.0'

## Data Pipeline

In [35]:
MNIST_ROOT = 'data'
INPUT_SIZE = 784
BATCH_SIZE = 10

In [36]:
class NoiseModifier(cdd.Modifier):
    def get(self, index):
        img, label = self._dataset[index]
        img += torch.rand_like(img) * 0.1
        img = torch.clip(img, 0, 255)
        return img, label


train_ds = torchvision.datasets.MNIST(root=MNIST_ROOT,
                                     train=True,
                                     transform=F.to_tensor,
                                     download=True)
test_ds = torchvision.datasets.MNIST(root=MNIST_ROOT,
                                    train=False,
                                    transform=F.to_tensor)

train_ds = cdd.Wrapper(train_ds)
train_ds.describe("This is MNIST dataset of handwritten images, TRAIN PART")
test_ds = cdd.Wrapper(test_ds)

train_ds = NoiseModifier(train_ds)
test_ds = NoiseModifier(test_ds)

train_dl = torch.utils.data.DataLoader(dataset=train_ds,
                                       batch_size=BATCH_SIZE,
                                       shuffle=True)
test_dl = torch.utils.data.DataLoader(dataset=test_ds,
                                      batch_size=BATCH_SIZE,
                                      shuffle=False)

## Module definition

In [37]:
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, *args, **kwargs):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.l1 = nn.Linear(input_size, hidden_size)
        self.l2 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, y):
         out = self.l1(y)
         out = self.relu(out)
         out = self.l2(out)

         return out

## Cascade wrapper

In [38]:
class Classifier(TorchModel):
    # In train we copy-paste regular pytorch trainloop,
    # but use self._model, where our SimpleNN is placed
    def fit(self, train_dl, num_epochs, lr, *args, **kwargs):
        criterion = nn.CrossEntropyLoss()
        optim = torch.optim.Adam(self._model.parameters(), lr=lr)

        ds_size = len(train_dl)
        for epoch in range(num_epochs):
            for x, (imgs, labels) in enumerate(train_dl):
                imgs = imgs.reshape(-1, self._model.input_size)

                out = self._model(imgs)
                loss = criterion(out, labels)

                optim.zero_grad()
                loss.backward()
                optim.step()

                if x % 500 == 0:
                    print (
                        f'Epochs [{epoch}/{num_epochs}], '
                        f'Step[{x}/{ds_size}], Loss: {loss.item():.4f}'
                    )

    # Evaluate function takes the metrics from arguments
    # and populates self.metrics without returning anything
    def evaluate(self, test_dl, metrics, *args, **kwargs):
        pred = []
        gt = []
        for imgs, labels in tqdm(test_dl):
            imgs = imgs.reshape(-1, self._model.input_size)
            out = torch.argmax(self._model(imgs, *args, **kwargs), -1)

            pred.append(out)
            gt.append(labels)

        pred = torch.concat(pred).detach().numpy()
        gt = torch.concat(gt).detach().numpy()

        for metric in metrics:
            metric.compute(gt, pred)
            self.add_metric(metric)

## Model training
Now we are ready to train our model. We define hyperparameters and pass them to our wrapper. Wrapper accepts pytorch module's class and all the parameters that are needed to initialize it.  
Additionally we pass keyword arguments that are connected to training. It is done to add them to the model's metadata.

In [39]:
NUM_EPOCHS = 2
LR = 1e-3

# Classifier will initialize SimpleNN with all the parameters passed
# but some of them are not for the SimpleNN, but to be recorded in metadata
model = Classifier(SimpleNN,
    # These arguments are needed by SimpleNN,
    # but passed as keywords to be recorded in meta
    input_size=INPUT_SIZE,
    hidden_size=100,
    num_classes=10,
    # These arguments will be skipped by SimpleNN,
    # but will be added to meta
    num_epochs=NUM_EPOCHS,
    lr=LR,
    bs=BATCH_SIZE)
model.fit(train_dl, NUM_EPOCHS, LR)

Epochs [0/2], Step[0/6000], Loss: 2.2989
Epochs [0/2], Step[500/6000], Loss: 0.3711
Epochs [0/2], Step[1000/6000], Loss: 0.1683
Epochs [0/2], Step[1500/6000], Loss: 0.2686
Epochs [0/2], Step[2000/6000], Loss: 0.7121
Epochs [0/2], Step[2500/6000], Loss: 0.2036
Epochs [0/2], Step[3000/6000], Loss: 0.0670
Epochs [0/2], Step[3500/6000], Loss: 0.0669
Epochs [0/2], Step[4000/6000], Loss: 0.0153
Epochs [0/2], Step[4500/6000], Loss: 0.1202
Epochs [0/2], Step[5000/6000], Loss: 0.5884
Epochs [0/2], Step[5500/6000], Loss: 0.0096
Epochs [1/2], Step[0/6000], Loss: 0.4009
Epochs [1/2], Step[500/6000], Loss: 0.4634
Epochs [1/2], Step[1000/6000], Loss: 0.5688
Epochs [1/2], Step[1500/6000], Loss: 0.2074
Epochs [1/2], Step[2000/6000], Loss: 0.0176
Epochs [1/2], Step[2500/6000], Loss: 0.0115
Epochs [1/2], Step[3000/6000], Loss: 0.0111
Epochs [1/2], Step[3500/6000], Loss: 0.2187
Epochs [1/2], Step[4000/6000], Loss: 0.1725
Epochs [1/2], Step[4500/6000], Loss: 0.0978
Epochs [1/2], Step[5000/6000], Loss: 0.0

## Evaluate the model
Now we can evaluate model performance on test dataset. We pass the data and one metric that is a wrapper around sklearn's metric.

In [40]:
model.evaluate(test_dl, [SkMetric("accuracy_score")])

100%|██████████| 1000/1000 [00:02<00:00, 402.95it/s]


## Check the metadata
Let's examine metadata obtained from the model after training.

In [41]:
from pprint import pprint
pprint(model.get_meta())

[{'comments': [],
  'created_at': DateTime(2026, 9, 4, 12, 59, 38, 856029, tzinfo=Timezone('UTC')),
  'description': None,
  'links': [],
  'metrics': [SkMetric(name=accuracy_score, value=0.9695, created_at=2026-09-04 13:00:27.328694+00:00)],
  'module': 'SimpleNN(\n'
            '  (l1): Linear(in_features=784, out_features=100, bias=True)\n'
            '  (l2): Linear(in_features=100, out_features=10, bias=True)\n'
            '  (relu): ReLU()\n'
            ')',
  'name': '__main__.Classifier',
  'params': {'bs': 10,
             'hidden_size': 100,
             'input_size': 784,
             'lr': 0.001,
             'num_classes': 10,
             'num_epochs': 2},
  'tags': [],
  'type': 'model'}]


We can notice several things. The model is tracking the time of creation. It's metrics in place as expected after evaluation.  
Let's look at the params dict. We can see all the parameters that we passed using keywords in the wrapper. The wrapper recorded them in the metadata for us automatically.

## Saving the model
It's time to save the trained model. We can just use model.save() method, but let's look at another Cascade's tool for model management.  
Model containers are hierarchically organized in the following way `Workspace` -> `Repo` -> (`ModelLine`/`DataLine`)

In [42]:
from cascade.workspaces import Workspace

line = Workspace("main") \
    .add_repo("tutorial") \
    .add_line('linear_nn', type="model")

This is the repository of models. It manages a series of experiments over a sets of models of different architectures called model lines.

In [43]:
line

<class 'cascade.lines.model_line.ModelLine'>(0) items of <class 'cascade.models.model.Model'>

Model line is the manager of models with similar architecture, but different parameters or different epochs. It manages saving of model and its meta and also loading of model.

Aside from model's metadata we would like to know on what data model was trained.

In [44]:
model.link(train_ds)
model.get_meta()

[{'name': '__main__.Classifier',
  'description': None,
  'tags': [],
  'comments': [],
  'links': [{'id': '1',
    'name': '__main__.NoiseModifier',
    'uri': None,
    'meta': [{'name': '__main__.NoiseModifier',
      'description': None,
      'tags': [],
      'comments': [],
      'links': [],
      'type': 'dataset',
      'data_card': None,
      'len': 60000},
     {'name': 'cascade.data.dataset.Wrapper',
      'description': 'This is MNIST dataset of handwritten images, TRAIN PART',
      'tags': [],
      'comments': [],
      'links': [],
      'type': 'dataset',
      'data_card': None,
      'len': 60000,
      'obj_type': "<class 'torchvision.datasets.mnist.MNIST'>"}],
    'created_at': DateTime(2026, 9, 4, 13, 1, 57, 862777, tzinfo=Timezone('UTC'))}],
  'type': 'model',
  'created_at': DateTime(2026, 9, 4, 12, 59, 38, 856029, tzinfo=Timezone('UTC')),
  'metrics': [SkMetric(name=accuracy_score, value=0.9695, created_at=2026-09-04 13:00:27.328694+00:00)],
  'params': {'in

We are ready to save the model

In [45]:
line.save(model)

This will save the model to the path:  
`repo/linear_nn/00000/model`  
And metadata:  
`repo/linear_nn/00000/meta.json`

## Peeking inside the repo

The easiest way to see inside Cascade repo is to run a query in terminal like this. after `query` command you can list the meta fields using pure Python. For example metrics is the list and we can address each value the same way as we would in plain Python.

In [51]:
!cd main && cascade query slug 'metrics[0].name' 'metrics[0].value'

──────────────────────────────────────────────────────────────────────────────
slug                      metrics[0].name           metrics[0].value          
──────────────────────────────────────────────────────────────────────────────
gifted_vivid_woodpecker   accuracy_score            0.9695                    
daffy_crane_of_abundance  accuracy_score            0.915                     
bizarre_venerable_bittern accuracy_score            0.9614                    
emerald_centipede_of_expreaccuracy_score            0.965                     
──────────────────────────────────────────────────────────────────────────────
Finished: 2026-09-04 16:06:06.015590+03:00
Returned rows: 4
Time: 0.0037s


## More experiments
What if we want to automatically run a number of experiments and then choose the best model?  
The workflow is pretty similar. In the example below we try to find the best option for hidden size of the model.  
We define the set of parameters for our experiments and run them in loop every time saving the results.

In [49]:
params = [
    {'hidden_size': 10,  'num_epochs': 2, 'lr': 0.001, 'bs': 10},
    {'hidden_size': 50,  'num_epochs': 2, 'lr': 0.001, 'bs': 10},
    {'hidden_size': 100, 'num_epochs': 2, 'lr': 0.001, 'bs': 10}
]

In [50]:
for p in params:
    model = Classifier(SimpleNN,
        **p,
        input_size=INPUT_SIZE,
        num_classes=10)
    model.fit(train_dl, **p)
    model.evaluate(test_dl, [SkMetric("accuracy_score")])
    repo['linear_nn'].save(model)

Epochs [0/2], Step[0/6000], Loss: 2.2516
Epochs [0/2], Step[500/6000], Loss: 0.8069
Epochs [0/2], Step[1000/6000], Loss: 0.4820
Epochs [0/2], Step[1500/6000], Loss: 0.1748
Epochs [0/2], Step[2000/6000], Loss: 0.2000
Epochs [0/2], Step[2500/6000], Loss: 1.1389
Epochs [0/2], Step[3000/6000], Loss: 0.3355
Epochs [0/2], Step[3500/6000], Loss: 0.4010
Epochs [0/2], Step[4000/6000], Loss: 0.7420
Epochs [0/2], Step[4500/6000], Loss: 0.1194
Epochs [0/2], Step[5000/6000], Loss: 0.4762
Epochs [0/2], Step[5500/6000], Loss: 0.4701
Epochs [1/2], Step[0/6000], Loss: 0.3466
Epochs [1/2], Step[500/6000], Loss: 0.2711
Epochs [1/2], Step[1000/6000], Loss: 0.0649
Epochs [1/2], Step[1500/6000], Loss: 0.3857
Epochs [1/2], Step[2000/6000], Loss: 0.1902
Epochs [1/2], Step[2500/6000], Loss: 0.1132
Epochs [1/2], Step[3000/6000], Loss: 0.3022
Epochs [1/2], Step[3500/6000], Loss: 0.3928
Epochs [1/2], Step[4000/6000], Loss: 0.5902
Epochs [1/2], Step[4500/6000], Loss: 0.5458
Epochs [1/2], Step[5000/6000], Loss: 0.3

100%|██████████| 1000/1000 [00:02<00:00, 432.42it/s]


Epochs [0/2], Step[0/6000], Loss: 2.3139
Epochs [0/2], Step[500/6000], Loss: 0.7037
Epochs [0/2], Step[1000/6000], Loss: 0.1377
Epochs [0/2], Step[1500/6000], Loss: 0.2905
Epochs [0/2], Step[2000/6000], Loss: 1.1446
Epochs [0/2], Step[2500/6000], Loss: 0.1073
Epochs [0/2], Step[3000/6000], Loss: 0.0893
Epochs [0/2], Step[3500/6000], Loss: 1.2855
Epochs [0/2], Step[4000/6000], Loss: 0.4909
Epochs [0/2], Step[4500/6000], Loss: 0.2973
Epochs [0/2], Step[5000/6000], Loss: 0.4350
Epochs [0/2], Step[5500/6000], Loss: 0.7323
Epochs [1/2], Step[0/6000], Loss: 0.0370
Epochs [1/2], Step[500/6000], Loss: 0.0874
Epochs [1/2], Step[1000/6000], Loss: 0.0159
Epochs [1/2], Step[1500/6000], Loss: 0.1280
Epochs [1/2], Step[2000/6000], Loss: 0.2406
Epochs [1/2], Step[2500/6000], Loss: 0.5819
Epochs [1/2], Step[3000/6000], Loss: 0.0178
Epochs [1/2], Step[3500/6000], Loss: 0.0103
Epochs [1/2], Step[4000/6000], Loss: 0.1398
Epochs [1/2], Step[4500/6000], Loss: 0.0806
Epochs [1/2], Step[5000/6000], Loss: 0.0

100%|██████████| 1000/1000 [00:02<00:00, 430.93it/s]


Epochs [0/2], Step[0/6000], Loss: 2.2967
Epochs [0/2], Step[500/6000], Loss: 0.6697
Epochs [0/2], Step[1000/6000], Loss: 0.1806
Epochs [0/2], Step[1500/6000], Loss: 0.0778
Epochs [0/2], Step[2000/6000], Loss: 0.0392
Epochs [0/2], Step[2500/6000], Loss: 0.0470
Epochs [0/2], Step[3000/6000], Loss: 0.0899
Epochs [0/2], Step[3500/6000], Loss: 0.0655
Epochs [0/2], Step[4000/6000], Loss: 0.0425
Epochs [0/2], Step[4500/6000], Loss: 0.8037
Epochs [0/2], Step[5000/6000], Loss: 0.0595
Epochs [0/2], Step[5500/6000], Loss: 0.0317
Epochs [1/2], Step[0/6000], Loss: 0.0049
Epochs [1/2], Step[500/6000], Loss: 0.0403
Epochs [1/2], Step[1000/6000], Loss: 0.2770
Epochs [1/2], Step[1500/6000], Loss: 0.1292
Epochs [1/2], Step[2000/6000], Loss: 0.0095
Epochs [1/2], Step[2500/6000], Loss: 0.0216
Epochs [1/2], Step[3000/6000], Loss: 0.0223
Epochs [1/2], Step[3500/6000], Loss: 0.0191
Epochs [1/2], Step[4000/6000], Loss: 0.0327
Epochs [1/2], Step[4500/6000], Loss: 0.0428
Epochs [1/2], Step[5000/6000], Loss: 0.0

100%|██████████| 1000/1000 [00:02<00:00, 404.10it/s]


Now we can run more advanced query with sorting our results by metric value

In [54]:
!cd main && cascade query slug 'metrics[0].name' 'metrics[0].value' sort 'metrics[0].value' desc

──────────────────────────────────────────────────────────────────────────────
slug                      metrics[0].name           metrics[0].value          
──────────────────────────────────────────────────────────────────────────────
gifted_vivid_woodpecker   accuracy_score            0.9695                    
emerald_centipede_of_expreaccuracy_score            0.965                     
bizarre_venerable_bittern accuracy_score            0.9614                    
daffy_crane_of_abundance  accuracy_score            0.915                     
──────────────────────────────────────────────────────────────────────────────
Finished: 2026-09-04 16:07:17.228246+03:00
Returned rows: 4
Time: 0.0104s
